In [1]:
import numpy as np
import torch
import torchmetrics
import torchvision
from scipy import _distributor_init
from sklearn.datasets import load_sample_images

sample_images = np.stack(load_sample_images()["images"])
sample_images = torch.tensor(sample_images, dtype=torch.float32) / 255

In [2]:
sample_images.shape

torch.Size([2, 427, 640, 3])

In [3]:
sample_images_permuted = sample_images.permute(0, 3, 1, 2)
sample_images_permuted.shape

torch.Size([2, 3, 427, 640])

In [4]:
import torchvision
import torchvision.transforms.v2 as T
cropped_images = T.CenterCrop((70, 120))(sample_images_permuted)
cropped_images.shape

torch.Size([2, 3, 70, 120])

In [5]:
import torch.nn as nn

torch.manual_seed(42)
conv_layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7)
fmaps = conv_layer(cropped_images)

In [6]:
fmaps.shape

torch.Size([2, 32, 64, 114])

In [7]:
conv_layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7, padding="same")
fmaps = conv_layer(cropped_images)
fmaps.shape

torch.Size([2, 32, 70, 120])

In [8]:
conv_layer.weight.shape

torch.Size([32, 3, 7, 7])

In [9]:
conv_layer.bias.shape

torch.Size([32])

In [10]:
max_pool = nn.MaxPool2d(kernel_size=2)
avg_pool = nn.AvgPool2d(kernel_size=2)

In [11]:
import torch.functional as F

class DepthPool(nn.Module):
    def __init__(self, kernel_size, stride=None, padding=0):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride if stride is not None else kernel_size
        self.padding = padding


    def forward(self, inputs):
        batch, channels, height, width = inputs.shape
        Z = inputs.view(batch, channels, height * width)  # merge spatial dims
        Z = Z.permute(0, 2, 1)  # switch spatial and channels dims
        Z = F.max_pool1d(Z, kernel_size=self.kernel_size, stride=self.stride,
                         padding=self.padding)  # compute max pool
        Z = Z.permute(0, 2, 1)  # switch back spatial and channels dims
        return Z.view(batch, -1, height, width)  # unmerge spatial dims

In [12]:
global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1)
output = global_avg_pool(cropped_images)

In [13]:
output = cropped_images.mean(dim=(2, 3), keepdim=True)

In [14]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [15]:
from functools import partial

DefaultConv2d = partial(nn.Conv2d, kernel_size=3, padding="same")

model = nn.Sequential(
    DefaultConv2d(in_channels=1, out_channels=64, kernel_size=7), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(64, 128), nn.ReLU(),
    DefaultConv2d(128, 128), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(128, 256), nn.ReLU(),
    DefaultConv2d(256, 256), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Flatten(),
    nn.Linear(2304, 128), nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(64, 47)  # EMNIST balanced has 47 classes
).to(device)

In [16]:
import torchmetrics
from torch.utils.data import DataLoader
from Neural_Networks_Deep_Learning.CIFAR10.utils import train

transform = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])
train_and_valid_data = torchvision.datasets.EMNIST(
    root="datasets", split="balanced", train=True, transform=transform)

test_data = torchvision.datasets.EMNIST(
    root="datasets", split="balanced", train=False, transform=transform)

train_data, valid_data = torch.utils.data.random_split(train_and_valid_data, [107800, 5000])

train_loader = DataLoader(train_data,  batch_size=32, num_workers=4, pin_memory=True, shuffle=True)
valid_loader = DataLoader(valid_data,  batch_size=32, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_data,   batch_size=32, num_workers=4, pin_memory=True)

n_epochs = 100
optimizer = torch.optim.AdamW(model.parameters())
criterion = nn.CrossEntropyLoss()
metric = torchmetrics.Accuracy(task="multiclass", num_classes=47).to(device)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer=optimizer, max_lr=1e-2, total_steps=len(train_loader)*n_epochs)
n_iter_no_improvements = 3
train(model, optimizer, criterion, train_loader, valid_loader, metric, n_epochs, n_iter_no_improvements, scheduler=scheduler)

Epoch: 1/100, Loss: 1.8130, Val Score: 0.8094
Epoch: 2/100, Loss: 0.9168, Val Score: 0.8320
Epoch: 3/100, Loss: 0.7484, Val Score: 0.8582
Epoch: 4/100, Loss: 0.6675, Val Score: 0.8566
Epoch: 5/100, Loss: 0.6236, Val Score: 0.8602


Exception in thread Thread-15 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/home/denys/PycharmProjects/Hands-On_ML/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 59, in _pin_memory_loop
    do_one_step()
  File "/home/denys/PycharmProjects/Hands-On_ML/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 35, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/denys/PycharmProjects/Hands-On_ML/.venv/lib/python3.12/site-packages/torch/multiprocessing/reductions.py", line 541, in rebuild_s

KeyboardInterrupt: 

In [17]:
from Neural_Networks_Deep_Learning.CIFAR10.utils import evaluate

evaluate(model, test_loader, metric)

0.8659042716026306

In [18]:
class SeparableConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super().__init__()
        self.depthwise_conv = nn.Conv2d(
            in_channels, in_channels, kernel_size, stride=stride,
            padding=padding, groups=in_channels)
        self.pointwise_conv = nn.Conv2d(
            in_channels, out_channels, kernel_size=1, stride=1, padding=0)

    def forward(self, inputs):
        return self.pointwise_conv(self.depthwise_conv(inputs))

In [19]:
class ResidualUnit(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        DefaultConv2d = partial(
            nn.Conv2d, kernel_size=3, stride=1, padding=1, bias=False)
        self.main_layers = nn.Sequential(
            DefaultConv2d(in_channels, out_channels, stride=stride),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            DefaultConv2d(out_channels, out_channels),
            nn.BatchNorm2d(out_channels),
        )
        if stride > 1:
            self.skip_connection = nn.Sequential(
                DefaultConv2d(in_channels, out_channels, kernel_size=1,
                              stride=stride, padding=0),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.skip_connection = nn.Identity()

    def forward(self, inputs):
        return F.relu(self.main_layers(inputs) + self.skip_connection(inputs))

In [20]:
class ResNet34(nn.Module):
    def __init__(self):
        super().__init__()
        layers = [
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, stride=2,
                      padding=3, bias=False),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        ]
        prev_filters = 64
        for filters in [64] * 3 + [128] * 4 + [256] * 6 + [512] * 3:
            stride = 1 if filters == prev_filters else 2
            layers.append(ResidualUnit(prev_filters, filters, stride=stride))
            prev_filters = filters
        layers += [
            nn.AdaptiveAvgPool2d(output_size=1),
            nn.Flatten(),
            nn.LazyLinear(10),
        ]
        self.resnet = nn.Sequential(*layers)

    def forward(self, inputs):
        return self.resnet(inputs)

In [21]:
weights = torchvision.models.ConvNeXt_Base_Weights.IMAGENET1K_V1
model = torchvision.models.convnext_base(weights=weights).to(device)

In [22]:
transforms = weights.transforms()
preprocessed_images = transforms(sample_images_permuted)

In [38]:
weights = torchvision.models.ConvNeXt_Base_Weights.IMAGENET1K_V1
transforms = weights.transforms()
print(transforms)

ImageClassification(
    crop_size=[224]
    resize_size=[232]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [23]:
model.eval()
with torch.no_grad():
    y_logits = model(preprocessed_images.to(device))

In [24]:
y_pred = torch.argmax(y_logits, dim=1)
y_pred

tensor([698, 985], device='cuda:0')

In [25]:
class_names = weights.meta["categories"]
[class_names[class_id] for class_id in y_pred]

['palace', 'daisy']

In [26]:
y_top3_logits, y_top3_class_ids = y_logits.topk(k=3, dim=1)
[[class_names[class_id] for class_id in top3] for top3 in y_top3_class_ids]

[['palace', 'monastery', 'lakeside'], ['daisy', 'pot', 'ant']]

In [27]:
y_top3_logits.softmax(dim=1)

tensor([[0.8618, 0.1185, 0.0197],
        [0.8106, 0.0964, 0.0930]], device='cuda:0')

In [28]:
DefaultFlowers102 = partial(torchvision.datasets.Flowers102, root="datasets",
                            transform=weights.transforms(), download=True)
train_set = DefaultFlowers102(split="train")
valid_set = DefaultFlowers102(split="val")
test_set = DefaultFlowers102(split="test")

In [29]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=32)
test_loader = DataLoader(test_set, batch_size=32)

In [30]:
#For this dataset there is no formatted labels
# class_names = ['pink primrose', ..., 'trumpet creeper', 'blackberry lily']

In [31]:
[name for name, child in model.named_children()]

['features', 'avgpool', 'classifier']

In [32]:
model.classifier

Sequential(
  (0): LayerNorm2d((1024,), eps=1e-06, elementwise_affine=True)
  (1): Flatten(start_dim=1, end_dim=-1)
  (2): Linear(in_features=1024, out_features=1000, bias=True)
)

In [33]:
n_classes=102
model.classifier[2] = nn.Linear(1024, n_classes).to(device)

In [34]:
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

In [35]:
from Neural_Networks_Deep_Learning.CIFAR10.utils import train

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
metric = torchmetrics.Accuracy(task="multiclass", num_classes=102).to(device)

train(model, optimizer, criterion, train_loader, valid_loader, metric, 10, 3)

Epoch: 1/10, Loss: 4.2843, Val Score: 0.6275
Epoch: 2/10, Loss: 2.9976, Val Score: 0.7676
Epoch: 3/10, Loss: 2.0514, Val Score: 0.8225
Epoch: 4/10, Loss: 1.3589, Val Score: 0.8529
Epoch: 5/10, Loss: 0.9493, Val Score: 0.8735
Epoch: 6/10, Loss: 0.6571, Val Score: 0.8765
Epoch: 7/10, Loss: 0.5236, Val Score: 0.8873
Epoch: 8/10, Loss: 0.3845, Val Score: 0.8863
Epoch: 9/10, Loss: 0.3096, Val Score: 0.8941
Epoch: 10/10, Loss: 0.2376, Val Score: 0.8882


0.8941176533699036

In [40]:
from Neural_Networks_Deep_Learning.CIFAR10.utils import evaluate

evaluate(model, test_loader, metric)

0.8754268884658813

In [39]:
import torchvision.transforms.v2 as T

transforms = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=30),
    T.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0)),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [43]:
train_set = torchvision.datasets.Flowers102(root="datasets", transform=transforms,
                                            download=True, split="train")

train_loader = DataLoader(train_set, batch_size=32, num_workers=4, pin_memory=True)

train(model, optimizer, criterion, train_loader, valid_loader, metric, 15, 3)

Epoch: 1/15, Loss: 0.6683, Val Score: 0.8745
Epoch: 2/15, Loss: 0.4750, Val Score: 0.8902
Epoch: 3/15, Loss: 0.3739, Val Score: 0.8990
Epoch: 4/15, Loss: 0.2908, Val Score: 0.9000
Epoch: 5/15, Loss: 0.2495, Val Score: 0.9010
Epoch: 6/15, Loss: 0.2416, Val Score: 0.9039
Epoch: 7/15, Loss: 0.1786, Val Score: 0.9029
Epoch: 8/15, Loss: 0.1694, Val Score: 0.9069
Epoch: 9/15, Loss: 0.1583, Val Score: 0.9049
Epoch: 10/15, Loss: 0.1407, Val Score: 0.9069
Epoch: 11/15, Loss: 0.1384, Val Score: 0.9108
Epoch: 12/15, Loss: 0.1249, Val Score: 0.9098
Epoch: 13/15, Loss: 0.1143, Val Score: 0.9029
Epoch: 14/15, Loss: 0.1185, Val Score: 0.9078
Validation score has not improved for 3 epochs, stopping training


0.9107843041419983

In [44]:
class FlowerLocation(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        self.localization_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(base_model.classifier[2].in_features, 4)
        )

    def forward(self, X):
        features = self.base_model.features(X)
        pool = self.base_model.avg_pool(features)
        logits = self.base_model.classifier(pool)
        bbox = self.localization_head(pool)

        return logits, bbox

torch.manual_seed(42)
locator_model = FlowerLocation(model).to(device)

In [45]:
#Training requires marked boundaries of object in image, which are not provided in FLower102
# preproc_images = [...]  # a batch of preprocessed images
# y_pred_logits, y_pred_bbox = locator_model(preprocessed_images.to(device))

In [46]:
import torchvision.tv_tensors

bbox = torchvision.tv_tensors.BoundingBoxes(
    [[377, 199, 248, 262]],
    format="CXCYWH",
    canvas_size=(500, 754)
)

In [74]:
transforms(bbox)

BoundingBoxes([[105,  90, 110, 150]], format=BoundingBoxFormat.CXCYWH, canvas_size=(224, 224))

In [77]:
first_image = [...]  # load the first training image without any preprocessing
preproc_image, preproc_target = transforms(
    (first_image, {"label": 0, "bbox": bbox})
)
preproc_bbox = preproc_target["bbox"]